# ⭐ Day 53: Decision Trees Explained  
## Gini vs Entropy | Complete Tutorial with Examples & Exercises

**Day 53 of 369-day Python & AI Learning Path** 🚀


## Welcome to Day 53! 🌳

Decision Trees are one of the most intuitive and powerful algorithms in machine learning. Unlike the "black box" nature of neural networks, Decision Trees offer transparency—you can actually see how decisions are being made, step by step, just like a flowchart!

Today we dive deep into how they work, understand the mathematical beauty behind splitting criteria, and master the difference between **Gini Impurity** and **Entropy** (Information Gain). Whether you're preparing for interviews, competitions, or production systems, understanding Decision Trees is absolutely essential.

By the end of this notebook, you'll not only understand the theory but also know exactly when to use Gini vs Entropy, how to prevent overfitting, and how to extract powerful insights from feature importance. Let's grow our knowledge tree! 🌱


## 📋 Table of Contents

1. [Introduction to Decision Trees](#1-introduction-to-decision-trees-and-why-they-matter)
2. [How Decision Trees Make Decisions](#2-how-decision-trees-make-decisions)
3. [Key Concepts: Nodes and Splitting](#3-key-concepts)
4. [Gini Impurity - Mathematical Deep Dive](#4-gini-impurity)
5. [Entropy & Information Gain](#5-entropy--information-gain)
6. [Gini vs Entropy - Detailed Comparison](#6-gini-vs-entropy)
7. [Building a Decision Tree from Scratch](#7-building-from-scratch)
8. [Scikit-Learn Implementation](#8-scikit-learn-implementation)
9. [Titanic Dataset Application](#9-titanic-dataset)
10. [Tree Visualization](#10-visualizing-the-tree)
11. [Hyperparameters Tuning](#11-hyperparameters)
12. [Overfitting and Control](#12-overfitting)
13. [Feature Importance](#13-feature-importance)
14. [Hands-On Exercises](#hands-on-exercises)
15. [Solutions](#solutions)


## 1. Introduction to Decision Trees and Why They Matter 💡

Decision Trees are supervised learning algorithms used for both classification and regression tasks. They work by recursively splitting the data into subsets based on feature values, creating a tree-like structure of decisions.

**Why Decision Trees are Awesome:**
- ✅ **Interpretable**: You can visualize and explain decisions
- ✅ **Handles non-linear relationships** naturally
- ✅ **No feature scaling required**
- ✅ **Handles both numerical and categorical data**
- ✅ **Foundation for Random Forests and Gradient Boosting**

**Real-world Applications:**
- Medical diagnosis (symptom-based decision paths)
- Credit scoring (loan approval decisions)
- Customer churn prediction
- Fraud detection systems


## 2. How Decision Trees Make Decisions 🔀

Imagine you're playing "20 Questions" to guess an animal. You ask questions like:
- "Does it have fur?" → Yes/No
- "Is it bigger than a cat?" → Yes/No
- "Does it live in water?" → Yes/No

Each question splits the possibilities until you narrow down to the answer. That's exactly how Decision Trees work!

**The Algorithm's Goal:** Find the best question (feature + threshold) that creates the purest subsets (homogeneous classes).


## 3. Key Concepts 📊

### Tree Structure Components:

**Root Node**: The topmost node representing the entire dataset

**Decision Nodes (Internal Nodes)**: Nodes that split the data based on a condition

**Leaf Nodes (Terminal Nodes)**: Final nodes that provide the prediction/class

**Splitting Criteria**: The metric used to decide how to split the data (Gini or Entropy)

**Pruning**: Removing branches to prevent overfitting


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set style for beautiful visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print("📦 NumPy version:", np.__version__)
print("📦 Pandas version:", pd.__version__)


## 4. Gini Impurity - Mathematical Deep Dive 📐

### Formula:
$$Gini = 1 - \sum_{i=1}^{C} (p_i)^2$$

Where:
- $p_i$ = proportion of class $i$ in the node
- $C$ = number of classes

### Interpretation:
- **Gini = 0**: Perfect purity (all samples belong to one class)
- **Gini = 0.5** (binary): Maximum impurity (50-50 split)
- Lower Gini = Better split


In [ ]:
def calculate_gini(proportions):
    """
    Calculate Gini Impurity for given class proportions.
    proportions: list of probabilities that sum to 1
    """
    gini = 1 - sum([p**2 for p in proportions])
    return gini

# Example calculations
examples = [
    ([1.0, 0.0], "Perfect purity (Class A only)"),
    ([0.5, 0.5], "Maximum impurity (50-50 split)"),
    ([0.8, 0.2], "Mostly Class A"),
    ([0.7, 0.3], "Skewed toward Class A"),
    ([0.9, 0.1], "Highly pure"),
]

print("🔢 Gini Impurity Calculations:\n")
for props, desc in examples:
    gini = calculate_gini(props)
    print(f"{desc}")
    print(f"   Proportions: {props}")
    print(f"   Gini = 1 - ({props[0]}² + {props[1]}²) = {gini:.4f}\n")


In [ ]:
# Visualize Gini Impurity across different probability distributions
p_values = np.linspace(0, 1, 100)
gini_values = [calculate_gini([p, 1-p]) for p in p_values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(p_values, gini_values, linewidth=3, color='#E74C3C', label='Gini Impurity')
ax.fill_between(p_values, gini_values, alpha=0.3, color='#E74C3C')

# Mark key points
ax.plot(0.5, 0.5, 'ko', markersize=10, label='Maximum Impurity (0.5)')
ax.plot(0, 0, 'go', markersize=10, label='Perfect Purity (0)')
ax.plot(1, 0, 'go', markersize=10)

ax.set_xlabel('Probability of Class A', fontsize=12)
ax.set_ylabel('Gini Impurity', fontsize=12)
ax.set_title('Gini Impurity vs Class Distribution\n(Binary Classification)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

print("📊 Observation: Gini is symmetric and peaks at 0.5 (maximum uncertainty)")


## 5. Entropy & Information Gain 🧮

### Entropy Formula:
$$Entropy = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

### Information Gain:
$$IG = Entropy_{parent} - \sum_{j} \frac{n_j}{n} Entropy_{child_j}$$

**Interpretation:**
- **Entropy = 0**: Perfect purity
- **Entropy = 1** (binary, 50-50): Maximum uncertainty
- Higher Information Gain = Better split


In [ ]:
def calculate_entropy(proportions):
    """
    Calculate Entropy for given class proportions.
    Uses log base 2 (units: bits)
    """
    entropy = 0
    for p in proportions:
        if p > 0:  # Avoid log(0)
            entropy -= p * np.log2(p)
    return entropy

def calculate_information_gain(parent_props, left_props, right_props, left_weight, right_weight):
    """
    Calculate Information Gain for a split.
    """
    parent_entropy = calculate_entropy(parent_props)
    left_entropy = calculate_entropy(left_props)
    right_entropy = calculate_entropy(right_props)
    
    weighted_child_entropy = left_weight * left_entropy + right_weight * right_entropy
    info_gain = parent_entropy - weighted_child_entropy
    
    return info_gain, parent_entropy, weighted_child_entropy

# Example calculations
print("🔢 Entropy Calculations:\n")
for props, desc in examples:
    entropy = calculate_entropy(props)
    gini = calculate_gini(props)
    print(f"{desc}")
    print(f"   Proportions: {props}")
    print(f"   Entropy = {entropy:.4f}")
    print(f"   Gini = {gini:.4f}\n")


In [ ]:
# Compare Gini vs Entropy side by side
p_values = np.linspace(0.001, 0.999, 100)
gini_values = [calculate_gini([p, 1-p]) for p in p_values]
entropy_values = [calculate_entropy([p, 1-p]) for p in p_values]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gini plot
ax1.plot(p_values, gini_values, linewidth=3, color='#E74C3C', label='Gini')
ax1.fill_between(p_values, gini_values, alpha=0.2, color='#E74C3C')
ax1.set_xlabel('Probability of Class A', fontsize=11)
ax1.set_ylabel('Impurity', fontsize=11)
ax1.set_title('Gini Impurity', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

# Entropy plot
ax2.plot(p_values, entropy_values, linewidth=3, color='#3498DB', label='Entropy')
ax2.fill_between(p_values, entropy_values, alpha=0.2, color='#3498DB')
ax2.set_xlabel('Probability of Class A', fontsize=11)
ax2.set_ylabel('Entropy (bits)', fontsize=11)
ax2.set_title('Entropy', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("📊 Key Insight: Both measures peak at 0.5 (maximum uncertainty)")
print("📊 Entropy has steeper curves near purity, making it more sensitive to changes")


In [ ]:
# Direct comparison overlay
fig, ax = plt.subplots(figsize=(10, 6))

# Normalize entropy to [0,0.5] for fair comparison with Gini
entropy_normalized = [e / 2 for e in entropy_values]

ax.plot(p_values, gini_values, linewidth=3, color='#E74C3C', label='Gini Impurity', linestyle='-')
ax.plot(p_values, entropy_normalized, linewidth=3, color='#3498DB', label='Entropy (normalized /2)', linestyle='--')

ax.set_xlabel('Probability of Class A', fontsize=12)
ax.set_ylabel('Impurity Measure', fontsize=12)
ax.set_title('Gini vs Entropy: Direct Comparison\n(Normalized for Scale)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

print("📊 Observation: When normalized, both curves are very similar!")
print("📊 Gini is slightly more concave, Entropy more sensitive near extremes")


## 6. Gini vs Entropy - Detailed Comparison ⚖️

| Aspect | Gini Impurity | Entropy (Information Gain) |
|--------|---------------|------------------------------|
| **Formula** | $1 - \sum p_i^2$ | $-\sum p_i \log_2(p_i)$ |
| **Computation** | Faster (no logarithms) | Slower (log calculations) |
| **Sensitivity** | Less sensitive to changes | More sensitive near purity |
| **Range** | [0, 0.5] for binary | [0, 1] for binary |
| **Bias** | Tends to select larger branches | Tends to select more balanced splits |
| **Use Case** | Default in sklearn, large datasets | When interpretability matters, smaller datasets |

### When to Use Which?

**Use Gini when:**
- 🚀 Training speed is critical
- 📊 Working with large datasets
- 🎯 You want slightly more aggressive pruning

**Use Entropy when:**
- 🔍 You need maximum information theoretical purity
- 📈 Working with smaller, cleaner datasets
- 🧠 You want more balanced tree structures

**Practical Advice:** Try both and cross-validate! The difference is usually minimal.


In [ ]:
# Practical comparison: Train identical trees with different criteria
from sklearn.datasets import make_classification

# Create synthetic dataset
X, y = make_classification(n_samples=1000, n_features=4, n_redundant=0, 
                          n_informative=4, n_clusters_per_class=1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train with Gini
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
tree_gini.fit(X_train, y_train)
gini_score = tree_gini.score(X_test, y_test)

# Train with Entropy
tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
tree_entropy.fit(X_train, y_train)
entropy_score = tree_entropy.score(X_test, y_test)

print("⚖️ Criteria Comparison on Synthetic Data:\n")
print(f"🎯 Gini Accuracy: {gini_score:.4f}")
print(f"🎯 Entropy Accuracy: {entropy_score:.4f}")
print(f"📊 Difference: {abs(gini_score - entropy_score):.4f}")
print("\n💡 Usually the difference is negligible - both work well!")


## 7. Building a Decision Tree from Scratch (Simplified) 🛠️

Let's implement a basic Decision Tree to understand the algorithm mechanics.


In [ ]:
class SimpleDecisionTree:
    """
    Simplified Decision Tree for educational purposes.
    Only supports binary classification and numerical features.
    """
    def __init__(self, max_depth=3, criterion='gini'):
        self.max_depth = max_depth
        self.criterion = criterion
        self.tree = None
    
    def _impurity(self, y):
        """Calculate impurity (Gini or Entropy)"""
        if len(y) == 0:
            return 0
        p = np.mean(y)
        if self.criterion == 'gini':
            return 2 * p * (1 - p)  # Simplified for binary
        else:
            if p == 0 or p == 1:
                return 0
            return -(p * np.log2(p) + (1-p) * np.log2(1-p))
    
    def _information_gain(self, y, left_idx, right_idx):
        """Calculate information gain from a split"""
        parent_impurity = self._impurity(y)
        n = len(y)
        n_left, n_right = len(left_idx), len(right_idx)
        
        if n_left == 0 or n_right == 0:
            return 0
        
        child_impurity = (n_left/n) * self._impurity(y[left_idx]) + \
                        (n_right/n) * self._impurity(y[right_idx])
        
        return parent_impurity - child_impurity
    
    def _best_split(self, X, y):
        """Find the best feature and threshold to split on"""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_idx = np.where(X[:, feature] <= threshold)[0]
                right_idx = np.where(X[:, feature] > threshold)[0]
                
                gain = self._information_gain(y, left_idx, right_idx)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the tree"""
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # Stopping criteria
        if (depth >= self.max_depth or n_classes == 1 or n_samples < 2):
            return {'prediction': np.bincount(y).argmax(), 'samples': n_samples}
        
        feature, threshold, gain = self._best_split(X, y)
        
        if gain <= 0:
            return {'prediction': np.bincount(y).argmax(), 'samples': n_samples}
        
        left_idx = np.where(X[:, feature] <= threshold)[0]
        right_idx = np.where(X[:, feature] > threshold)[0]
        
        return {
            'feature': feature,
            'threshold': threshold,
            'gain': gain,
            'samples': n_samples,
            'left': self._build_tree(X[left_idx], y[left_idx], depth + 1),
            'right': self._build_tree(X[right_idx], y[right_idx], depth + 1)
        }
    
    def fit(self, X, y):
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_one(self, x, node):
        """Predict single sample"""
        if 'prediction' in node:
            return node['prediction']
        
        if x[node['feature']] <= node['threshold']:
            return self._predict_one(x, node['left'])
        else:
            return self._predict_one(x, node['right'])
    
    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

# Test our implementation
print("🌳 Testing Custom Decision Tree Implementation...")
custom_tree = SimpleDecisionTree(max_depth=3, criterion='gini')
custom_tree.fit(X_train, y_train)
custom_pred = custom_tree.predict(X_test)
custom_accuracy = accuracy_score(y_test, custom_pred)

print(f"✅ Custom Tree Accuracy: {custom_accuracy:.4f}")
print(f"✅ Sklearn Tree Accuracy: {gini_score:.4f}")
print("\n💡 Our simplified implementation works! (May differ slightly due to implementation details)")


## 8. Scikit-Learn DecisionTreeClassifier 🚀

Now let's use the industry-standard implementation with all its optimizations.


In [ ]:
# Comprehensive Decision Tree with Scikit-Learn
from sklearn.tree import export_text

# Create a more complex dataset for demonstration
X_demo, y_demo = make_classification(n_samples=500, n_features=5, n_informative=3, 
                                    n_redundant=0, n_classes=2, random_state=42)

feature_names = ['Feature_A', 'Feature_B', 'Feature_C', 'Feature_D', 'Feature_E']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.25, random_state=42)

# Initialize and train
dt_classifier = DecisionTreeClassifier(
    criterion='gini',      # or 'entropy'
    max_depth=4,           # Prevent overfitting
    min_samples_split=10,  # Minimum samples to split
    min_samples_leaf=5,    # Minimum samples in leaf
    random_state=42
)

dt_classifier.fit(X_train_d, y_train_d)

# Predictions
y_pred = dt_classifier.predict(X_test_d)
y_pred_proba = dt_classifier.predict_proba(X_test_d)

# Evaluation
print("📊 Decision Tree Performance:\n")
print(f"✅ Accuracy: {accuracy_score(y_test_d, y_pred):.4f}")
print(f"📈 Training Score: {dt_classifier.score(X_train_d, y_train_d):.4f}")
print(f"📉 Test Score: {dt_classifier.score(X_test_d, y_test_d):.4f}")
print("\n🔍 Classification Report:")
print(classification_report(y_test_d, y_pred, target_names=['Class 0', 'Class 1']))


## 9. Applying to Titanic Dataset 🚢

Let's apply Decision Trees to the classic Titanic survival prediction problem.


In [ ]:
# Load Titanic dataset (using seaborn's built-in dataset)
titanic = sns.load_dataset('titanic')

print("🚢 Titanic Dataset Overview:")
print(f"Shape: {titanic.shape}")
print("\n📋 First few rows:")
print(titanic.head())
print("\n📊 Missing values:")
print(titanic.isnull().sum())
print(f"\n🎯 Survival rate: {titanic['survived'].mean():.2%}")


In [ ]:
# Preprocessing Pipeline
def preprocess_titanic(df):
    """Clean and preprocess Titanic data"""
    data = df.copy()
    
    # Select relevant features
    features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
    data = data[features + ['survived']]
    
    # Handle missing values
    data['age'].fillna(data['age'].median(), inplace=True)
    data['embarked'].fillna(data['embarked'].mode()[0], inplace=True)
    data['fare'].fillna(data['fare'].median(), inplace=True)
    
    # Encode categorical variables
    le_sex = LabelEncoder()
    data['sex'] = le_sex.fit_transform(data['sex'])
    
    le_embarked = LabelEncoder()
    data['embarked'] = le_embarked.fit_transform(data['embarked'])
    
    return data

titanic_clean = preprocess_titanic(titanic)

# Prepare features and target
X_titanic = titanic_clean.drop('survived', axis=1)
y_titanic = titanic_clean['survived']

feature_names_titanic = X_titanic.columns.tolist()

print("✅ Preprocessing complete!")
print(f"Features: {feature_names_titanic}")
print(f"Dataset shape: {X_titanic.shape}")
print("\n📊 Cleaned data sample:")
print(X_titanic.head())


In [ ]:
# Split and train
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic)

# Train Decision Tree
titanic_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

titanic_tree.fit(X_train_t, y_train_t)

# Predictions
y_pred_t = titanic_tree.predict(X_test_t)

# Evaluation
print("🚢 Titanic Survival Prediction Results:\n")
print(f"🎯 Test Accuracy: {accuracy_score(y_test_t, y_pred_t):.4f}")
print(f"📈 Training Accuracy: {titanic_tree.score(X_train_t, y_train_t):.4f}")
print("\n📋 Detailed Classification Report:")
print(classification_report(y_test_t, y_pred_t, target_names=['Did Not Survive', 'Survived']))


## 10. Visualizing the Tree 🎨

One of the best features of Decision Trees is interpretability!


In [ ]:
# Visualize the Titanic Decision Tree
fig, ax = plt.subplots(figsize=(20, 12))

plot_tree(titanic_tree, 
          feature_names=feature_names_titanic,
          class_names=['Did Not Survive', 'Survived'],
          filled=True,
          rounded=True,
          fontsize=10,
          ax=ax,
          impurity=True,
          proportion=True)

plt.title('🚢 Titanic Survival Decision Tree\n(Gini Impurity, Max Depth=5)', 
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("🎨 Tree Visualization Complete!")
print("\n💡 Color intensity indicates purity (darker = more pure)")
print("💡 Each node shows: feature, threshold, gini, samples, class distribution")


In [ ]:
# Compare Gini vs Entropy trees side by side
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Gini tree
tree_gini_viz = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree_gini_viz.fit(X_train_t, y_train_t)

plot_tree(tree_gini_viz, 
          feature_names=feature_names_titanic,
          class_names=['Did Not Survive', 'Survived'],
          filled=True, rounded=True, fontsize=9, ax=axes[0])
axes[0].set_title('Gini Impurity\n(Max Depth=3)', fontsize=14, fontweight='bold')

# Entropy tree
tree_entropy_viz = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
tree_entropy_viz.fit(X_train_t, y_train_t)

plot_tree(tree_entropy_viz, 
          feature_names=feature_names_titanic,
          class_names=['Did Not Survive', 'Survived'],
          filled=True, rounded=True, fontsize=9, ax=axes[1])
axes[1].set_title('Entropy (Information Gain)\n(Max Depth=3)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("⚖️ Side-by-side comparison of splitting criteria!")
print(f"Gini Accuracy: {tree_gini_viz.score(X_test_t, y_test_t):.4f}")
print(f"Entropy Accuracy: {tree_entropy_viz.score(X_test_t, y_test_t):.4f}")


## 11. Hyperparameters Tuning ⚙️

Key hyperparameters to control tree complexity:


In [ ]:
# Hyperparameter exploration
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 10, 20, 50],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}

# Use GridSearchCV for systematic search (limited for demo)
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={
        'max_depth': [3, 5, 7, 10],
        'min_samples_split': [2, 10, 20],
        'criterion': ['gini', 'entropy']
    },
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_t, y_train_t)

print("⚙️ Hyperparameter Tuning Results:\n")
print(f"🎯 Best Parameters: {grid_search.best_params_}")
print(f"🎯 Best CV Score: {grid_search.best_score_:.4f}")
print(f"🎯 Test Score: {grid_search.score(X_test_t, y_test_t):.4f}")

# Display all results
results_df = pd.DataFrame(grid_search.cv_results_)
print("\n📊 Top 5 Configurations:")
print(results_df.nlargest(5, 'mean_test_score')[['param_criterion', 'param_max_depth', 
                                                'param_min_samples_split', 'mean_test_score']])


## 12. Overfitting in Decision Trees 🛑

Decision Trees are prone to overfitting. Let's visualize this effect.


In [ ]:
# Demonstrate overfitting with varying max_depth
max_depths = range(1, 21)
train_scores = []
test_scores = []
cv_scores = []

for depth in max_depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_t, y_train_t)
    
    train_scores.append(dt.score(X_train_t, y_train_t))
    test_scores.append(dt.score(X_test_t, y_test_t))
    cv_scores.append(cross_val_score(dt, X_train_t, y_train_t, cv=5).mean())

# Plot
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(max_depths, train_scores, 'o-', label='Training Score', linewidth=2, markersize=6, color='#2ECC71')
ax.plot(max_depths, test_scores, 's-', label='Test Score', linewidth=2, markersize=6, color='#E74C3C')
ax.plot(max_depths, cv_scores, '^-', label='CV Score (5-fold)', linewidth=2, markersize=6, color='#3498DB')

ax.set_xlabel('Max Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Overfitting in Decision Trees\n(Training vs Test Performance)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.05)

# Annotate optimal point
optimal_depth = max_depths[np.argmax(test_scores)]
ax.axvline(optimal_depth, color='purple', linestyle='--', alpha=0.7, label=f'Optimal Depth: {optimal_depth}')
ax.annotate(f'Optimal: {optimal_depth}', xy=(optimal_depth, max(test_scores)), 
            xytext=(optimal_depth+2, max(test_scores)-0.05),
            arrowprops=dict(arrowstyle='->', color='purple'),
            fontsize=10, color='purple')

plt.tight_layout()
plt.show()

print(f"📊 Optimal max_depth: {optimal_depth}")
print(f"🎯 Best test accuracy: {max(test_scores):.4f}")
print("\n⚠️  Notice how training score keeps increasing while test score plateaus!")


## 13. Feature Importance 📈

Decision Trees provide built-in feature importance based on how much each feature reduces impurity.


In [ ]:
# Feature importance analysis
importances = titanic_tree.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(feature_names_titanic)))

bars = ax.bar(range(len(importances)), importances[indices], color=colors)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feature_names_titanic[i] for i in indices], rotation=45, ha='right')
ax.set_ylabel('Importance', fontsize=12)
ax.set_title('🌟 Feature Importance in Titanic Survival Prediction\n(Gini Importance)', 
             fontsize=14, fontweight='bold')

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("📊 Feature Importance Ranking:\n")
for i in indices:
    print(f"{i+1}. {feature_names_titanic[i]}: {importances[i]:.4f}")

print("\n💡 Interpretation: Higher values mean the feature is more important for predictions")


In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test_t, y_pred_t)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'],
            annot_kws={'size': 14})
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title('Confusion Matrix - Titanic Predictions', fontsize=14, fontweight='bold')

# Add percentage annotations
total = cm.sum()
for i in range(2):
    for j in range(2):
        percentage = cm[i, j] / total * 100
        ax.text(j+0.5, i+0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

print("📊 Confusion Matrix Analysis:")
print(f"True Negatives: {cm[0,0]} | False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]} | True Positives: {cm[1,1]}")


## 🛠️ Hands-On Exercises

Now it's your turn! Complete these exercises to master Decision Trees.

### Exercise 1: Manual Gini Calculation
Calculate the Gini impurity for a node with 30 samples: 20 Class A and 10 Class B.


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Manual Entropy Calculation
Calculate the Entropy for the same node (20 Class A, 10 Class B). Then calculate Information Gain if splitting creates two child nodes: Left (15 A, 5 B) and Right (5 A, 5 B).


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Train with Both Criteria
Train two Decision Trees on the Iris dataset (load using `sklearn.datasets.load_iris()`), one with Gini and one with Entropy. Compare their accuracies.


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Visualize Both Trees
Visualize both trees from Exercise 3 side by side and observe if they make different splits.


In [ ]:
# Exercise 4: Your code here



### Exercise 5: Hyperparameter Tuning
Experiment with different `max_depth` values (2, 5, 10, 20, None) on the Titanic dataset. Plot training and test scores to identify overfitting.


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Feature Importance Analysis
Train a Decision Tree on the Boston Housing dataset (regression) or Wine dataset (classification) and identify the top 3 most important features.


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Control Overfitting
Using the Titanic dataset, find the best combination of `min_samples_split` and `min_samples_leaf` using cross-validation.


In [ ]:
# Exercise 7: Your code here



### Exercise 8: Compare with Logistic Regression
Compare Decision Tree performance with Logistic Regression on the Titanic dataset. Which performs better? Why?


In [ ]:
# Exercise 8: Your code here



### Exercise 9: Tree Pruning
Implement cost-complexity pruning (use `ccp_alpha` parameter) and find the optimal pruning level for the Titanic dataset.


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Real-world Application
Choose a dataset from Kaggle or sklearn (e.g., Heart Disease, Customer Churn) and build a complete Decision Tree pipeline with preprocessing, tuning, and evaluation.


In [ ]:
# Exercise 10: Your code here



## Solutions (Review After Attempting) ✅

Below are detailed solutions. Try the exercises first before checking!


In [ ]:
# Solution 1: Manual Gini Calculation
"""
Gini = 1 - (p_A² + p_B²)
p_A = 20/30 = 0.667
p_B = 10/30 = 0.333
Gini = 1 - (0.667² + 0.333²) = 1 - (0.444 + 0.111) = 1 - 0.555 = 0.444
"""

p_A, p_B = 20/30, 10/30
gini = 1 - (p_A**2 + p_B**2)
print(f"Solution 1 - Gini Impurity: {gini:.4f}")


In [ ]:
# Solution 2: Entropy and Information Gain
"""
Parent Entropy = -(0.667*log2(0.667) + 0.333*log2(0.333)) = 0.918 bits
Left Child (15A, 5B): p_A=0.75, p_B=0.25, Entropy = 0.811
Right Child (5A, 5B): p_A=0.5, p_B=0.5, Entropy = 1.0
Weighted Child Entropy = (20/30)*0.811 + (10/30)*1.0 = 0.874
Information Gain = 0.918 - 0.874 = 0.044 bits
"""

parent_entropy = calculate_entropy([20/30, 10/30])
left_entropy = calculate_entropy([15/20, 5/20])
right_entropy = calculate_entropy([5/10, 5/10])
weighted_child = (20/30)*left_entropy + (10/30)*right_entropy
info_gain = parent_entropy - weighted_child

print(f"Solution 2 - Parent Entropy: {parent_entropy:.4f}")
print(f"Solution 2 - Information Gain: {info_gain:.4f}")


In [ ]:
# Solution 3 & 4: Iris Dataset Comparison
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# Train both
tree_g = DecisionTreeClassifier(criterion='gini', random_state=42)
tree_e = DecisionTreeClassifier(criterion='entropy', random_state=42)
tree_g.fit(X_train_i, y_train_i)
tree_e.fit(X_train_i, y_train_i)

print("Solution 3 - Iris Dataset Results:")
print(f"Gini Accuracy: {tree_g.score(X_test_i, y_test_i):.4f}")
print(f"Entropy Accuracy: {tree_e.score(X_test_i, y_test_i):.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_tree(tree_g, feature_names=iris.feature_names, class_names=iris.target_names, 
          filled=True, ax=axes[0], max_depth=3)
axes[0].set_title('Gini Tree', fontsize=12)
plot_tree(tree_e, feature_names=iris.feature_names, class_names=iris.target_names, 
          filled=True, ax=axes[1], max_depth=3)
axes[1].set_title('Entropy Tree', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Solution 5: Overfitting Demonstration
depths = [2, 3, 5, 7, 10, 15, 20, None]
train_s, test_s = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_t, y_train_t)
    train_s.append(dt.score(X_train_t, y_train_t))
    test_s.append(dt.score(X_test_t, y_test_t))

plt.figure(figsize=(10, 6))
plt.plot([str(d) for d in depths], train_s, 'o-', label='Train', linewidth=2)
plt.plot([str(d) for d in depths], test_s, 's-', label='Test', linewidth=2)
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Solution 5: Overfitting Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Solution 5: Notice test score drops after depth=5 (overfitting begins)")


In [ ]:
# Solution 6: Wine Dataset Feature Importance
from sklearn.datasets import load_wine

wine = load_wine()
X_w, y_w = wine.data, wine.target
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_w, y_w, test_size=0.3, random_state=42)

dt_wine = DecisionTreeClassifier(random_state=42)
dt_wine.fit(X_train_w, y_train_w)

importance_df = pd.DataFrame({
    'feature': wine.feature_names,
    'importance': dt_wine.feature_importances_
}).sort_values('importance', ascending=False)

print("Solution 6 - Top 3 Features:")
print(importance_df.head(3))

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'][:5], importance_df['importance'][:5])
plt.xlabel('Importance')
plt.title('Solution 6: Top 5 Wine Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Solution 7: Grid Search for Overfitting Control
param_grid = {
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10]
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train_t, y_train_t)

print("Solution 7 - Best Parameters:")
print(grid.best_params_)
print(f"Best CV Score: {grid.best_score_:.4f}")
print(f"Test Score: {grid.score(X_test_t, y_test_t):.4f}")


In [ ]:
# Solution 8: Compare with Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_t)
X_test_scaled = scaler.transform(X_test_t)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train_t)
lr_score = lr.score(X_test_scaled, y_test_t)

dt_score = titanic_tree.score(X_test_t, y_test_t)

print("Solution 8 - Model Comparison:")
print(f"Decision Tree: {dt_score:.4f}")
print(f"Logistic Regression: {lr_score:.4f}")
print("\nNote: Logistic Regression may perform better with limited data,")
print("while Decision Trees can capture non-linear patterns with more data.")


In [ ]:
# Solution 9: Cost-Complexity Pruning
path = titanic_tree.cost_complexity_pruning_path(X_train_t, y_train_t)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

train_scores_p, test_scores_p = [], []
for ccp_alpha in ccp_alphas[:10]:  # Limit for demo
    dt_pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    dt_pruned.fit(X_train_t, y_train_t)
    train_scores_p.append(dt_pruned.score(X_train_t, y_train_t))
    test_scores_p.append(dt_pruned.score(X_test_t, y_test_t))

plt.figure(figsize=(10, 6))
plt.plot(ccp_alphas[:10], train_scores_p, 'o-', label='Train')
plt.plot(ccp_alphas[:10], test_scores_p, 's-', label='Test')
plt.xlabel('ccp_alpha')
plt.ylabel('Accuracy')
plt.title('Solution 9: Cost-Complexity Pruning')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_alpha = ccp_alphas[np.argmax(test_scores_p)]
print(f"Solution 9 - Optimal ccp_alpha: {best_alpha:.6f}")


In [ ]:
# Solution 10: Complete Pipeline Template
"""
Complete Pipeline for Any Dataset:

1. Load and explore data
2. Handle missing values
3. Encode categorical variables
4. Split into train/test
5. Train Decision Tree with cross-validation
6. Tune hyperparameters (GridSearchCV)
7. Evaluate with multiple metrics
8. Visualize tree and feature importance
9. Analyze errors and iterate

Example structure:
"""

def complete_dt_pipeline(X, y, dataset_name="Dataset"):
    # Split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Tune
    grid = GridSearchCV(
        DecisionTreeClassifier(random_state=42),
        {'max_depth': [3,5,7,10], 'min_samples_split': [2,5,10]},
        cv=5, scoring='accuracy'
    )
    grid.fit(X_tr, y_tr)
    
    # Evaluate
    best_model = grid.best_estimator_
    train_acc = best_model.score(X_tr, y_tr)
    test_acc = best_model.score(X_te, y_te)
    cv_score = grid.best_score_
    
    print(f"📊 {dataset_name} Results:")
    print(f"Best params: {grid.best_params_}")
    print(f"CV Score: {cv_score:.4f}")
    print(f"Train: {train_acc:.4f} | Test: {test_acc:.4f}")
    
    return best_model

print("Solution 10 - Complete pipeline template provided above!")
print("Apply this structure to any dataset (Heart Disease, Churn, etc.)")


---

## 🎉 Congratulations!

Great work! You now deeply understand one of the most fundamental algorithms in machine learning. You've mastered:

- ✅ The mathematical foundations (Gini vs Entropy)
- ✅ How to build and visualize Decision Trees
- ✅ Hyperparameter tuning and overfitting control
- ✅ Feature importance interpretation
- ✅ Practical applications with real datasets

### What's Next? 🔮

**Tomorrow we explore Random Forest – The Power of Ensembles!** 

We'll learn how combining multiple Decision Trees can dramatically improve accuracy and robustness. Get ready to enter the world of ensemble methods!

---

*Python & AI Learning Path | Day 53 / 369* 🚀

**Keep learning, keep growing! 🌳→🌲→🌴**
